In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

/Users/oscar/Desktop/LangGraph/langgraph/myenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
load_dotenv()

True

In [3]:
model=ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9)

In [4]:
class BlogPostState(TypedDict): # type: ignore
    title: str
    outline: str
    blog_post: str
    rating: int

In [10]:
def generate_outline(state: BlogPostState) -> BlogPostState:
    title=state["title"]
    prompt = f"Generate a detailed outline for a blog post with the title: {title}"
    outline=model.invoke(prompt)
    state["outline"]=outline
    return state

In [11]:
def write_blog_post(state: BlogPostState) -> BlogPostState:
    title=state["title"]
    outline=state["outline"]
    prompt = f"Write a blog post based on title:{title} based on the following outline: {outline}"
    blog_post=model.invoke(prompt)
    state["blog_post"]=blog_post
    return state

In [18]:
def rate_blog_post(state: BlogPostState) -> BlogPostState:
    blog_post=state["blog_post"]
    # Ask the model to reply with a single integer between 1 and 10 to improve parsing reliability
    prompt = f"Rate the following blog post on a scale of 1 to 10 (reply with a single integer between 1 and 10 only): {blog_post}"
    # The model may still return surrounding text; parse the first integer and clamp it to 1..10
    rating_resp = model.invoke(prompt)
    import re
    text = str(rating_resp).strip()
    m = re.search(r"(\d+)", text)
    parsed = None
    if m:
        try:
            parsed = int(m.group(1))
        except Exception:
            parsed = None
    else:
        # Try a last-ditch numeric conversion (handles '8.0' etc.)
        try:
            parsed = int(float(text))
        except Exception:
            parsed = None
    # Enforce rating range 1..10. If parsing failed, default to 1.
    if parsed is None:
        parsed = 1
    rating = max(1, min(10, parsed))
    state["rating"] = rating
    return state

graph= StateGraph(BlogPostState)
graph.add_node("generate_outline",generate_outline)
graph.add_node("write_blog_post",write_blog_post)
graph.add_node("rate_blog_post",rate_blog_post)
graph.add_edge(START,"generate_outline")
graph.add_edge("generate_outline","write_blog_post")
graph.add_edge("write_blog_post","rate_blog_post")
graph.add_edge("rate_blog_post",END)
workflow = graph.compile()

initial_state = BlogPostState(title="The Future of AI", outline="", blog_post="", rating=0)
result = workflow.invoke(initial_state)
print("Generated Outline:\n", result["outline"])
print("\nGenerated Blog Post:\n", result["blog_post"])
print("\nBlog Post Rating:\n", result["rating"])



Generated Outline:
 content='I. Introduction\n    A. Explanation of AI and its current uses\n    B. Importance of discussing the future of AI\n    C. Preview of topics to be covered in the blog post\n\nII. Current advancements in AI\n    A. Overview of recent developments in AI technology\n    B. Examples of AI in everyday life, such as virtual assistants and self-driving cars\n    C. Potential benefits of AI in various industries, such as healthcare and finance\n\nIII. Challenges and ethical considerations\n    A. Concerns about job displacement due to automation\n    B. Privacy issues related to the collection of personal data\n    C. Potential bias in AI algorithms\n    D. Importance of developing ethical guidelines for AI technology\n\nIV. Future possibilities for AI\n    A. Prediction of AI advancements in the next decade\n    B. Potential applications of AI in fields such as education and environmental conservation\n    C. Discussion of the concept of Artificial General Intellige